In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
engine = create_engine("postgresql+psycopg2://postgres:Zxcvbnm,./123@localhost:5432/ecommerce")
#Overall Summary 
overall = pd.read_sql("""
SELECT 
    COUNT(*) AS total_orders,
    ROUND(SUM(net_sales)::numeric, 2) AS total_revenue,
    ROUND(SUM(profit)::numeric, 2) AS total_profit,
    ROUND(AVG(net_sales)::numeric, 2) AS avg_order_value,
    ROUND(AVG(profit_margin_percentage)::numeric, 2) AS avg_profit_margin
FROM sales;
""", engine)

# Region Summary
region = pd.read_sql("""
SELECT region,
       ROUND(SUM(net_sales)::numeric, 2) AS total_revenue,
       ROUND(SUM(profit)::numeric, 2) AS total_profit,
       COUNT(*) AS total_orders
FROM sales
GROUP BY region
ORDER BY total_revenue DESC;
""", engine)

# Customer Segment
segment = pd.read_sql("""
SELECT customer_segment,
       ROUND(SUM(net_sales)::numeric, 2) AS total_revenue,
       ROUND(SUM(profit)::numeric, 2) AS total_profit,
       COUNT(*) AS total_orders
FROM sales
GROUP BY customer_segment
ORDER BY total_revenue DESC;
""", engine)

print(overall)
print(region)
print(segment)


   total_orders  total_revenue  total_profit  avg_order_value  \
0        138116   1.771343e+08   76146395.76           1282.5   

   avg_profit_margin  
0              45.07  
    region  total_revenue  total_profit  total_orders
0    South    55587994.98   23343946.17         43990
1  Central    37171107.17   15572868.84         29627
2     West    34755096.47   15647252.47         26273
3     East    27054887.88   11179366.94         21602
4    North    22565177.24   10402961.34         16624
  customer_segment  total_revenue  total_profit  total_orders
0         Consumer    97201007.90   42819790.02         75636
1          Premium    44269794.13   18177727.78         34746
2              VIP    17903164.81    7366352.68         13957
3         Business    17760296.90    7782525.28         13777


In [3]:
# Sales Channel
channel = pd.read_sql("""
SELECT Sales_channel,
	ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
	ROUND(SUM(Profit)::numeric, 2) AS Total_Profit,
    COUNT(*) AS Total_Orders
FROM sales
GROUP BY Sales_channel
ORDER BY Sales_channel DESC;
""", engine)

# Marketing Channel
marketing = pd.read_sql("""
SELECT marketing_channel,
	ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
	ROUND(SUM(Profit)::numeric, 2) AS Total_Profit,
    COUNT(*) AS Total_Orders
FROM sales
GROUP BY marketing_channel
ORDER BY marketing_channel DESC;
""", engine)

# Repeat V/S New Customer
Repeat = pd.read_sql("""
SELECT is_repeat_customer, ROUND(SUM(profit)::numeric, 2) AS Total_Product_profit,
		ROUND(SUM(net_sales)::numeric, 2) AS Total_revenue,
	ROUND(SUM(Profit)::numeric, 2) AS Total_Profit,
	ROUND(AVG(net_sales)::numeric, 2) AS AVerage_Order_value,
    COUNT(*) AS Total_Orders	
FROM sales
GROUP BY is_repeat_customer;
""", engine)

print(channel)
print(marketing)
print(Repeat)


  sales_channel  total_revenue  total_profit  total_orders
0       Website    61854736.08   26639693.01         48237
1  Social Media    17748554.91    7641363.37         13863
2    Mobile App    70834782.53   30451112.22         55318
3   Marketplace    26696190.22   11414227.16         20698
  marketing_channel  total_revenue  total_profit  total_orders
0           YouTube     5197990.67    2240159.85          4080
1            TikTok     6980204.06    3011928.49          5437
2          Referral    14342124.81    6186481.45         11191
3    Organic Search    35658209.46   15313174.30         27742
4         Instagram    17676446.46    7585410.75         13801
5        Google Ads    26540586.44   11356389.55         20633
6      Facebook Ads    21372470.11    9189184.91         16651
7   Email Marketing    14420446.43    6177791.79         11218
8            Direct    26117885.52   11280316.74         20515
9         Affiliate     8827899.78    3805557.93          6848
   is_repeat

In [7]:
# Return Status
returns = pd.read_sql("""
SELECT return_status,
       COUNT(*) AS total_orders,
       ROUND(SUM(net_sales)::numeric, 2) AS total_revenue,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM sales
GROUP BY return_status
ORDER BY total_orders DESC;
""", engine)

# Discount Range
discount = discount = pd.read_sql(text("""
SELECT 
    CASE 
        WHEN discount_amount = 0 THEN '0%'
        WHEN discount_amount > 0 AND discount_amount <= 100 THEN 'Low (0-100)'
        WHEN discount_amount > 100 AND discount_amount <= 300 THEN 'Medium (100-300)'
        ELSE 'High (300+)'
    END AS discount_range,
    COUNT(*) AS total_orders,
    ROUND(SUM(net_sales)::numeric, 2) AS total_revenue,
    ROUND(SUM(profit)::numeric, 2) AS total_profit,
    ROUND(AVG(profit_margin_percentage)::numeric, 2) AS avg_profit_margin
FROM sales
GROUP BY discount_range
ORDER BY total_revenue DESC;
"""), engine)

print(returns)
print(discount)

  return_status  total_orders  total_revenue  percentage
0           NaN        128654   1.639616e+08       93.15
1      Returned          9462   1.317267e+07        6.85
     discount_range  total_orders  total_revenue  total_profit  \
0       High (300+)         32075    74130029.01   25623148.42   
1  Medium (100-300)         38103    50992440.04   23709915.49   
2       Low (0-100)         50074    27482249.90   13913358.33   
3                0%         17864    24529544.79   12899973.52   

   avg_profit_margin  
0              32.73  
1              44.88  
2              49.88  
3              54.11  


# Key Insights

- The business generated ₹177.13 Cr in revenue and ₹76.15 Cr in profit, with a healthy average profit margin of 45.07%.
- South is the highest-performing region (₹55.59 Cr), while North is the lowest (₹22.57 Cr).
- The Consumer segment contributes the largest share of revenue (₹97.20 Cr), significantly ahead of Premium, VIP, and Business segments.
- Mobile App is the top sales channel (₹70.83 Cr), followed by Website. Social Media is the weakest channel.
- Organic Search is the best marketing channel (₹35.66 Cr), followed by Direct and Google Ads. YouTube and TikTok contribute relatively less.
- Repeat customers account for almost all revenue (₹176.7 Cr). New customers are extremely limited (only 334 orders).
- Only 6.85% of orders were returned, which is a healthy figure.
- Orders with high discounts (₹300+) show the lowest profit margin (32.73%), while zero-discount orders have the highest margin (54.11%).


# Business Recommendations

- Focus on acquiring more new customers. Current growth is almost entirely dependent on repeat buyers.
- Invest more in Mobile App and Website, as these two channels drive the majority of revenue.
- Scale Organic Search and Google Ads, since they deliver strong returns.
- Improve performance in the North region through targeted campaigns.
- Optimize the discount strategy by reducing heavy discounting and focusing on low-to-medium discounts.
- Strengthen retention and upselling programs for the Consumer segment.

In [8]:
overall.to_csv("overall_metrics.csv", index=False)
region.to_csv("revenue_by_region.csv", index=False)
segment.to_csv("revenue_by_segment.csv", index=False)
channel.to_csv("revenue_by_sales_channel.csv", index=False)
marketing.to_csv("revenue_by_marketing_channel.csv", index=False)
Repeat.to_csv("repeat_vs_new_customers.csv", index=False)
returns.to_csv("return_status.csv", index=False)
discount.to_csv("discount_impact.csv", index=False)

print("All summary tables exported successfully!")

All summary tables exported successfully!


In [10]:
overall.to_csv("overall_metrics.csv", index=False)